In [ ]:
from PIL import Image

In [ ]:
INSTRUCTION = ("You are a robot navigating an enclosed space."
" Your goal is to navigate to the correct object based on the user's commands. You were given the following task by the"
" user '{TASK}'. Currently, you are facing a scene represented by the given image. Reason about what you are seeing,"
" comparing what you know about the task (given the user commands) and the given scene. For example, if the task is"
" 'Navigate to the black leather sofa near a lampstand' your reasoning process will be"
" 'I'm currently observing a brown sofa which is different than"
" black, making it unlikely to be the target sofa. Moreover, there"
" is no lampstand near it, only a rug and a window' etc. If there"
" are distortions or artifact, do not focus on them, focus on the"
" object at hand. At the end of the reasoning process, evaluate"
" how well the provided image aligns with the user's task. Assign"
" a confidence score based on the following scale: - 0: You are"
" certain the image DOES NOT match the task. - 1: You are unsure"
" whether the image matches the task or not. - 2: You are certain"
" the image DOES match the task. Provide a concise reasoning"
" (under 100 words) and strictly follow this output format:\n"
"<motivation>Your reasoning here</motivation><score>0, 1, or 2</score>")

MODEL_NAME = "e-zorzi/Qwen2.5-VL-7B-Instruct-tuned-final"
DATASET_NAME = "reasoning-augmentation/rubrics"

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME)
dataset

In [ ]:
NUM_ROW = 555

row = dataset['train'][NUM_ROW]
task = row['instruction']
image = row['image']
reasoning = row['reasoning']
score = row['score']

image.save("./image.png")

image, task, reasoning, score


In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info  # pip install qwen-vl-utils
import torch

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"  # swap for your checkpoint (local path or HF repo id)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,   # use torch.float16 if bf16 isn't supported
    device_map="cuda:0",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)


def generate(prompt, image_path=None, image_url=None, max_new_tokens=512):
    content = []
    if image_path:
        content.append({"type": "image", "image": f"file://{image_path}"})
    elif image_url:
        content.append({"type": "image", "image": image_url})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # strip prompt tokens, keep only newly generated ones
    trimmed = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)
    ]
    return processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]


if __name__ == "__main__":
    examples = [
        {"prompt": INSTRUCTION.format(TASK=task), "image_path": "./image.png"},
    ]

    for ex in examples:
        print("=" * 80)
        print(f"PROMPT: {ex['prompt']}")
        print("-" * 80)
        print(generate(**ex))

In [1]:
import os
import torch 
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
from monorepo import ClientBasedLLM

MODEL_NAME = "Qwen/Qwen3-0.6B" #"Qwen/Qwen3-30B-A3B"  # swap for your checkpoint (local path or HF repo id)

client = ClientBasedLLM(model_id=MODEL_NAME)


/scratch/ezorzi/grpo-robot-navigation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[WARN] `ClientBasedLLM` requires a connection with a local VLLM server. Make sure to run the command `vllm serve Qwen/Qwen3-0.6B <options>` in a terminal, and wait for its initialization.


In [6]:
client.ask(prompt="\\nothink Who are you?")

'<think>\n\n</think>\n\nI am Qwen, a large-scale language model developed by Alibaba Group. I can answer questions, create text, write code, and do many other things. How can I assist you today?'

In [7]:
from monorepo import GeminiLLM, load_api_keys

load_api_keys()

gemini = GeminiLLM(model_id="gemini-3.6-flash")


Loaded dotenv file at $HOME/.env.ml: True


In [12]:
gemini.ask(prompt="Tell me a story?")

'The town of Oakhaven smelled constantly of damp pine and woodsmoke, but inside the shop of Master Silas, it smelled of copper, sweet oil, and old paper. \n\nSilas was a clockmaker of rare talent. People said he could make a pendulum swing to the heartbeat of a sleeping robin. But Silas possessed one clock that he had never built, and certainly had never sold.\n\nIt was a silver pocket watch, smooth as a river pebble, with a single, unblemished ruby set into the winding crown. It did not tick. It had no face, no hands, and no gears inside. Yet, Silas kept it in his vest pocket, resting against his ribs like a second heart.\n\nThe watch was a family heirloom with a dangerous secret: if you pressed the ruby crown, time stopped. \n\nRaindrops would freeze in mid-air like glass beads. Flocks of birds would hang suspended above the rooftops. The world would turn quiet, grayscale, and perfectly still, allowing Silas to walk through the frozen moment for as long as he liked. \n\nThere was, ho